# Blinkit Data Analytics — Activation funnel
DuckDB SQL + Python | Yash Prajapati

## 1. Setup

### 1.1 Libraries

In [1]:
import warnings, math, textwrap
warnings.filterwarnings('ignore')
import duckdb, pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', lambda v: f'{v:,.2f}')
plt.rcParams.update({'figure.figsize':(10,4.5),'axes.grid':True,'grid.alpha':.25,'axes.spines.top':False,'axes.spines.right':False,'font.size':10})
print('duckdb', duckdb.__version__, '| pandas', pd.__version__, '| numpy', np.__version__)

duckdb 1.5.5 | pandas 3.0.2 | numpy 2.4.4


### 1.2 Load cleaned workbook

In [2]:
XLSX = 'data/Blinkit_analysis_new.xlsx'
book = pd.read_excel(XLSX, sheet_name=None)
geo = pd.read_csv('data/city_state_zone.csv')
for name, df in book.items():
    print(f'{name:28s} {df.shape[0]:>6,} rows  {df.shape[1]:>3} cols')

Data_Quality_Report              46 rows    3 cols
Orders_Customer_Info          5,000 rows   16 cols
Orders_Raw_Archive            5,000 rows   23 cols
Order_Line_Items              5,000 rows   14 cols
Delivery_Performance          5,000 rows    8 cols
Customer_Feedback             5,000 rows    8 cols
Customers                     2,500 rows   11 cols
Products                        268 rows   10 cols
Inventory_Analysis              268 rows   18 cols
Data_Dictionary                  44 rows    4 cols
Reference_Parameters             11 rows    5 cols
Relational_Diagram                0 rows    0 cols
Pivot_Payment_Method              7 rows    7 cols
Pivot_Customer_Segment            7 rows    5 cols
Pivot_Category_Sales             14 rows    4 cols
Pivot_Monthly_Trend              24 rows    6 cols
Pivot_Area_Delivery              23 rows    6 cols
Pivot_Inventory_Movement          6 rows    7 cols
Pivot_Category_Stock             14 rows    8 cols


### 1.3 Create DuckDB database and load tables

In [3]:
con = duckdb.connect('blinkit.duckdb')
load = {'orders_src':'Orders_Raw_Archive','items_src':'Order_Line_Items','delivery_src':'Delivery_Performance',
        'feedback_src':'Customer_Feedback','customers_src':'Customers','products_src':'Products','inventory_src':'Inventory_Analysis'}
for tbl, sheet in load.items():
    df = book[sheet].copy()
    df.columns = [c.strip() for c in df.columns]
    con.register('tmp_df', df)
    con.execute(f'CREATE OR REPLACE TABLE {tbl} AS SELECT * FROM tmp_df')
con.register('geo_df', geo)
con.execute('CREATE OR REPLACE TABLE geo AS SELECT * FROM geo_df')
con.execute("SELECT table_name, estimated_size FROM duckdb_tables() ORDER BY table_name").df()

,table_name,estimated_size
0,customers_src,2500
1,delivery_src,5000
2,feedback_src,5000
3,geo,316
4,inventory_src,268
5,items_src,5000
6,orders_src,5000
7,products_src,268


### 1.4 Typed analysis views

In [4]:
con.execute('''
CREATE OR REPLACE VIEW orders AS
SELECT order_id, customer_id,
       strptime(order_date, '%d-%m-%Y %H:%M')              AS order_ts,
       CAST(strptime(order_date, '%d-%m-%Y %H:%M') AS DATE) AS order_date,
       strptime(promised_delivery_time, '%d-%m-%Y %H:%M')   AS promised_ts,
       strptime(actual_delivery_time,  '%d-%m-%Y %H:%M')    AS actual_ts,
       delivery_status, order_total, payment_method, delivery_partner_id, store_id,
       CAST(delivery_time_minutes AS INTEGER)               AS delay_min,
       distance_km, reasons_if_delayed, customer_name,
       trim(area) AS area, pincode, customer_segment,
       CAST(registration_date AS DATE)                      AS registration_date,
       order_day_of_week, order_time_slot, order_value_segment,
       CASE WHEN is_weekend = 'Yes' THEN 1 ELSE 0 END       AS is_weekend
FROM orders_src ''')

con.execute('''
CREATE OR REPLACE VIEW items AS
SELECT order_id, product_id, quantity, unit_price, product_name, category, brand,
       price, mrp, margin_percentage, shelf_life_days, min_stock_level, max_stock_level, line_total,
       line_total * margin_percentage / 100.0 AS margin_value
FROM items_src ''')

con.execute('''
CREATE OR REPLACE VIEW feedback AS
SELECT feedback_id, order_id, customer_id, rating, feedback_category, sentiment,
       CAST(feedback_date AS DATE) AS feedback_date
FROM feedback_src ''')

con.execute('''
CREATE OR REPLACE VIEW f_sales AS
SELECT o.order_id, o.customer_id, o.order_ts, o.order_date,
       date_trunc('month', o.order_date)  AS order_month,
       extract(hour FROM o.order_ts)      AS order_hour,
       o.order_day_of_week, o.is_weekend, o.order_time_slot, o.order_value_segment,
       o.payment_method, o.customer_segment, o.registration_date, o.customer_name,
       o.area, g.state, g.zone, g.city_tier,
       i.product_id, i.product_name, i.category, i.brand,
       i.quantity, i.line_total AS revenue, i.margin_value, i.margin_percentage,
       i.price, i.mrp, i.shelf_life_days,
       o.delay_min, o.distance_km, o.delivery_status,
       CASE WHEN o.delivery_status = 'On Time' THEN 1 ELSE 0 END AS is_on_time_status,
       CASE WHEN o.delay_min > 0 THEN 1 ELSE 0 END               AS is_late_minutes,
       f.rating, f.sentiment, f.feedback_category
FROM orders o
JOIN items i    ON i.order_id = o.order_id
LEFT JOIN geo g ON g.area     = o.area
LEFT JOIN feedback f ON f.order_id = o.order_id ''')

con.execute('SELECT COUNT(*) AS fact_rows, COUNT(DISTINCT order_id) AS orders, MIN(order_date) AS first_day, MAX(order_date) AS last_day FROM f_sales').df()

,fact_rows,orders,first_day,last_day
0,5000,5000,2023-03-16,2024-11-04


### 1.5 Query helper

In [5]:
def q(sql, con=con):
    return con.execute(textwrap.dedent(sql)).df()

def pct(x, n):
    return round(100.0 * x / n, 2) if n else 0.0

q('SELECT COUNT(*) AS rows_in_fact_view FROM f_sales')

,rows_in_fact_view
0,5000


## 5. Activation funnel

### 5.1 Funnel from registration to third order

In [6]:
q('''
WITH c AS (
  SELECT cs.customer_id, COUNT(DISTINCT o.order_id) AS orders
  FROM customers_src cs LEFT JOIN orders o ON o.customer_id = cs.customer_id GROUP BY 1)
SELECT 'Registered'       AS stage, COUNT(*) AS customers, 100.0 AS pct_of_registered FROM c
UNION ALL SELECT 'Placed 1st order', COUNT(*) FILTER (WHERE orders >= 1), round(100.0 * COUNT(*) FILTER (WHERE orders >= 1) / COUNT(*), 2) FROM c
UNION ALL SELECT 'Placed 2nd order', COUNT(*) FILTER (WHERE orders >= 2), round(100.0 * COUNT(*) FILTER (WHERE orders >= 2) / COUNT(*), 2) FROM c
UNION ALL SELECT 'Placed 3rd order', COUNT(*) FILTER (WHERE orders >= 3), round(100.0 * COUNT(*) FILTER (WHERE orders >= 3) / COUNT(*), 2) FROM c
UNION ALL SELECT 'Placed 4th order or more', COUNT(*) FILTER (WHERE orders >= 4), round(100.0 * COUNT(*) FILTER (WHERE orders >= 4) / COUNT(*), 2) FROM c ''')

,stage,customers,pct_of_registered
0,Registered,2500,100.00
1,Placed 1st order,2172,86.88
2,Placed 2nd order,1492,59.68
3,Placed 3rd order,806,32.24
4,Placed 4th order or more,355,14.20


### 5.2 Drop-off between stages

In [7]:
funnel = q('''
WITH c AS (SELECT cs.customer_id, COUNT(DISTINCT o.order_id) AS orders
           FROM customers_src cs LEFT JOIN orders o ON o.customer_id = cs.customer_id GROUP BY 1)
SELECT COUNT(*) AS registered,
       COUNT(*) FILTER (WHERE orders >= 1) AS first_order,
       COUNT(*) FILTER (WHERE orders >= 2) AS second_order,
       COUNT(*) FILTER (WHERE orders >= 3) AS third_order FROM c ''').iloc[0]
steps = [('Registered to 1st order', funnel.registered, funnel.first_order),
         ('1st to 2nd order',        funnel.first_order, funnel.second_order),
         ('2nd to 3rd order',        funnel.second_order, funnel.third_order)]
pd.DataFrame([{'step':s,'from':int(a),'to':int(b),'lost':int(a-b),'drop_off_pct':pct(a-b,a)} for s,a,b in steps])

,step,from,to,lost,drop_off_pct
0,Registered to 1st order,2500,2172,328,13.12
1,1st to 2nd order,2172,1492,680,31.31
2,2nd to 3rd order,1492,806,686,45.98


### 5.3 Days from registration to first order

In [8]:
q('''
WITH f AS (
  SELECT customer_id, MIN(order_date) AS first_order, MIN(registration_date) AS reg FROM orders GROUP BY 1)
SELECT CASE WHEN date_diff('day', reg, first_order) < 0 THEN 'a. order before registration (data issue)'
            WHEN date_diff('day', reg, first_order) <= 7   THEN 'b. within 7 days'
            WHEN date_diff('day', reg, first_order) <= 30  THEN 'c. 8 to 30 days'
            WHEN date_diff('day', reg, first_order) <= 90  THEN 'd. 31 to 90 days'
            ELSE 'e. more than 90 days' END AS activation_speed,
       COUNT(*) AS customers,
       round(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS share_pct
FROM f GROUP BY 1 ORDER BY 1 ''')

,activation_speed,customers,share_pct
0,a. order before registration (data issue),1438,66.21
1,b. within 7 days,27,1.24
2,c. 8 to 30 days,78,3.59
3,d. 31 to 90 days,192,8.84
4,e. more than 90 days,437,20.12


### 5.4 Gap to second order in bands

In [9]:
q('''
WITH seq AS (
  SELECT customer_id, order_date,
         ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date) AS order_no,
         LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date) AS prev_date
  FROM (SELECT DISTINCT customer_id, order_date FROM f_sales)),
g AS (SELECT date_diff('day', prev_date, order_date) AS gap FROM seq WHERE order_no = 2)
SELECT CASE WHEN gap <= 30 THEN 'a. 0-30 days' WHEN gap <= 60 THEN 'b. 31-60 days'
            WHEN gap <= 90 THEN 'c. 61-90 days' WHEN gap <= 180 THEN 'd. 91-180 days'
            WHEN gap <= 365 THEN 'e. 181-365 days' ELSE 'f. over 365 days' END AS gap_band,
       COUNT(*) AS customers,
       round(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS share_pct
FROM g GROUP BY 1 ORDER BY 1 ''')

,gap_band,customers,share_pct
0,a. 0-30 days,197,13.22
1,b. 31-60 days,188,12.62
2,c. 61-90 days,156,10.47
3,d. 91-180 days,367,24.63
4,e. 181-365 days,438,29.40
5,f. over 365 days,144,9.66


### 5.5 Repeat purchase curve (Kaplan-Meier estimate)

In [10]:
first_second = q('''
WITH seq AS (
  SELECT customer_id, order_date,
         ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date) AS order_no
  FROM (SELECT DISTINCT customer_id, order_date FROM f_sales)),
f AS (SELECT customer_id, MIN(order_date) FILTER (WHERE order_no = 1) AS first_date,
             MIN(order_date) FILTER (WHERE order_no = 2) AS second_date
      FROM seq GROUP BY customer_id)
SELECT customer_id, first_date, second_date,
       (SELECT MAX(order_date) FROM f_sales) AS study_end FROM f ''')

first_second['duration'] = np.where(first_second.second_date.notna(),
    (pd.to_datetime(first_second.second_date) - pd.to_datetime(first_second.first_date)).dt.days,
    (pd.to_datetime(first_second.study_end) - pd.to_datetime(first_second.first_date)).dt.days)
first_second['event'] = first_second.second_date.notna().astype(int)

times = np.sort(first_second.loc[first_second.event == 1, 'duration'].unique())
n = len(first_second); surv = 1.0; km = []
for t in times:
    at_risk = (first_second.duration >= t).sum()
    events  = ((first_second.duration == t) & (first_second.event == 1)).sum()
    surv *= (1 - events / at_risk)
    km.append({'days': int(t), 'at_risk': int(at_risk), 'repeat_orders': int(events),
               'not_returned_prob': round(surv, 4), 'returned_prob': round(1 - surv, 4)})
km_df = pd.DataFrame(km)
km_df[km_df.days.isin([7, 14, 30, 60, 90, 120, 180, 270, 365])]

,days,at_risk,repeat_orders,not_returned_prob,returned_prob
6,7,2122,4,0.98,0.02
13,14,2063,6,0.95,0.05
29,30,1953,6,0.91,0.09
59,60,1726,5,0.82,0.18
89,90,1539,7,0.74,0.26
118,120,1356,7,0.67,0.33
175,180,1056,1,0.56,0.44
258,270,692,3,0.41,0.59
341,365,410,4,0.30,0.70
